In [1]:
!pip install transformers peft torch gymnasium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 117.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.1/721.1 kB 54.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 153.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 159.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 51.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 58.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.2/532.2 MB 57.6 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 79.4 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 101.9 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 99.5 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 112.9 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 101.0 MB/s  0:00:010:00:01

In [27]:
!pip install "trl<=0.11.0" --upgrade


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
!pip install "git+https://github.com/volatilemolotov/agent-sandbox.git@gymnasium-api#subdirectory=clients/python/agentic-sandbox-client" --upgrade

  Cloning https://github.com/volatilemolotov/agent-sandbox.git (to revision gymnasium-api) to /tmp/pip-req-build-acfe6fmq
  Running command git clone --filter=blob:none --quiet https://github.com/volatilemolotov/agent-sandbox.git /tmp/pip-req-build-acfe6fmq
  Running command git checkout -b gymnasium-api --track origin/gymnasium-api
  Switched to a new branch 'gymnasium-api'
  Branch 'gymnasium-api' set up to track remote branch 'gymnasium-api' from 'origin'.
  Resolved https://github.com/volatilemolotov/agent-sandbox.git to commit dd252768e777ce4d6d10c0009759ab8766e38143
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 15.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 26.3 MB/s  0:00:00
  Created wheel for k8s-agent-sandbox: filename=k8s_agent_sandbox-0.1.dev581+gdd252768e-py3-none-any.whl size=77499 sha256=89b

In [4]:
from k8s_agent_sandbox.sandbox_env import SandboxEnv, SparseTaskReward, StepPenaltyReward

env = SandboxEnv(
    reward_fn=StepPenaltyReward(
        base=SparseTaskReward(
            success_fn=lambda obs, info: "manage.py" in obs and info["exit_code"] == 0
        ),
        penalty=0.02,
    ),
    warmpool="simple-sandbox-warmpool",
    connection_mode="gateway",   # swap to "gateway" for GKE
    connection_cfg={
        "gateway_name": "external-http-gateway",
        "gateway_namespace": "default",
    },
    max_episode_steps=15,
)

agent_tasks = [
    "create a django poll project",
]

trajectory = []

for task in agent_tasks:
    obs, info = env.reset(options={"task": task})
    terminated, truncated = False, False

    while not terminated and not truncated:
        action = "ls -la" #agent.act(obs, task)
        obs, reward, terminated, truncated, info = env.step(action)
        print(f"""
obs={obs}
reward={reward}
terminated={terminated}
truncated={truncated}
info={info}
""")
        trajectory.append((obs, action, reward, info))

env.close()



obs=total 13
drwxr-xr-x 1 pwuser pwuser 4096 Jun 26 09:59 .
drwxr-xr-x 1 root   root   4096 Jun 26 09:59 ..
drwxr-xr-x 2 pwuser pwuser 4096 Jun 26 09:59 __pycache__
-rw-r--r-- 1 pwuser pwuser 1023 Jun 26 09:45 main.py

reward=-0.02
terminated=False
truncated=False
info={'exit_code': 0, 'stdout': 'total 13\ndrwxr-xr-x 1 pwuser pwuser 4096 Jun 26 09:59 .\ndrwxr-xr-x 1 root   root   4096 Jun 26 09:59 ..\ndrwxr-xr-x 2 pwuser pwuser 4096 Jun 26 09:59 __pycache__\n-rw-r--r-- 1 pwuser pwuser 1023 Jun 26 09:45 main.py\n', 'stderr': '', 'elapsed_ms': 170, 'step': 1}


obs=total 13
drwxr-xr-x 1 pwuser pwuser 4096 Jun 26 09:59 .
drwxr-xr-x 1 root   root   4096 Jun 26 09:59 ..
drwxr-xr-x 2 pwuser pwuser 4096 Jun 26 09:59 __pycache__
-rw-r--r-- 1 pwuser pwuser 1023 Jun 26 09:45 main.py

reward=-0.02
terminated=False
truncated=False
info={'exit_code': 0, 'stdout': 'total 13\ndrwxr-xr-x 1 pwuser pwuser 4096 Jun 26 09:59 .\ndrwxr-xr-x 1 root   root   4096 Jun 26 09:59 ..\ndrwxr-xr-x 2 pwuser pwuser 4

In [6]:
from k8s_agent_sandbox import SandboxClient
from k8s_agent_sandbox.models import SandboxGatewayConnectionConfig
import time

client = SandboxClient(
    connection_config=SandboxGatewayConnectionConfig(
        gateway_name="external-http-gateway",
        gateway_namespace="default",
        # server_port=80,
    )
)

sandbox = client.create_sandbox(warmpool="simple-sandbox-warmpool")

start = time.time()
try:
    result = sandbox.commands.run("ls -la")
    print("stdout:", result.stdout)
    print("stderr:", result.stderr)
    print("exit_code:", result.exit_code)
finally:
    sandbox.terminate()
print(time.time() - start)

stdout: total 13
drwxr-xr-x 1 pwuser pwuser 4096 Jun 26 10:40 .
drwxr-xr-x 1 root   root   4096 Jun 26 10:40 ..
drwxr-xr-x 2 pwuser pwuser 4096 Jun 26 10:40 __pycache__
-rw-r--r-- 1 pwuser pwuser 1023 Jun 26 09:45 main.py

stderr: 
exit_code: 0
0.12109851837158203


In [3]:
import torch
from transformers import AutoTokenizer
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer
from peft import LoraConfig

from k8s_agent_sandbox.sandbox_env import SandboxEnv, SparseTaskReward

# ── 1. Configuration & Setup ─────────────────────────────────────────────────

model_name = "Qwen/Qwen2.5-Coder-1.5B"  # Good starting point for code/bash
ppo_config = PPOConfig(
    learning_rate=1.41e-5,
    batch_size=1,           # Keep small for local testing
    mini_batch_size=1,
)

# Apply LoRA to make training memory-efficient
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

# ── 2. Load Model and Tokenizer ──────────────────────────────────────────────

# TRL uses a special wrapper that adds a "Value Head" to the LLM for PPO
model = AutoModelForCausalLMWithValueHead.from_pretrained(
    model_name,
    peft_config=lora_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Initialize the PPO Trainer
ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=model,
    ref_model=None, # TRL handles the reference model internally with PEFT
    tokenizer=tokenizer,
)

# ── 3. Initialize Your Custom Environment ────────────────────────────────────

# Let's use a task where the agent must create a specific file
env = SandboxEnv(
    reward_fn=SparseTaskReward(
        success_fn=lambda obs, info: "hello_world.sh" in obs and info["exit_code"] == 0
    ),
    warmpool="simple-sandbox-warmpool",
    connection_mode="tunnel",
    max_episode_steps=5,
)

# ── 4. The Training Loop ─────────────────────────────────────────────────────

epochs = 50
task = "Create a bash script named hello_world.sh that prints 'Hello World'."

for epoch in range(epochs):
    obs, info = env.reset(options={"task": task})
    terminated, truncated = False, False
    
    # We will collect the step data to update the model at the end of the episode
    episode_queries = []
    episode_responses = []
    episode_rewards = []

    while not terminated and not truncated:
        # A. Format the Prompt for the LLM
        prompt = f"System: You are an expert bash programmer.\nTask: {task}\nTerminal Output: {obs}\nCommand to run:"
        query_tensor = tokenizer(prompt, return_tensors="pt").input_ids.squeeze().to(ppo_trainer.accelerator.device)
        
        # B. Generate the Action (Shell Command)
        # We generate a short sequence representing the next bash command
        generation_kwargs = {"max_new_tokens": 32, "do_sample": True, "top_k": 0.0, "top_p": 1.0}
        response_tensor = ppo_trainer.generate(query_tensor, **generation_kwargs).squeeze()
        
        # C. Decode the Action and Step the Environment
        # Extract *only* the newly generated text
        action = tokenizer.decode(response_tensor[len(query_tensor):], skip_special_tokens=True).strip()
        print(f"Agent executed: {action}")
        
        next_obs, reward, terminated, truncated, info = env.step(action)
        print(f"Reward: {reward} | Exit Code: {info['exit_code']}")
        
        # D. Store tensors for PPO Update
        episode_queries.append(query_tensor)
        episode_responses.append(response_tensor[len(query_tensor):])
        episode_rewards.append(torch.tensor(reward, dtype=torch.float32))
        
        obs = next_obs
        
        # End episode early if successful
        if reward > 0:
            break

    # E. Perform the PPO Optimization Step
    # TRL expects lists of tensors for a batch update
    stats = ppo_trainer.step(episode_queries, episode_responses, episode_rewards)
    print(f"Epoch {epoch} finished. Loss: {stats['ppo/loss/total']}")

env.close()

# Save your fine-tuned model adapters!
model.save_pretrained("./sandbox-agent-lora")

/usr/local/lib/python3.11/dist-packages/trl/trainer/ppo_config.py:207: FutureWarning: `PPOConfig` is deprecated and will be removed in the future. Please use `PPOv2Config` with `PPOv2Trainer` instead.
  warnings.warn(


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/trl/trainer/ppo_trainer.py:193: FutureWarning: `PPOTrainer` is deprecated and will be removed in trl v0.12. Please use `PPOv2Trainer` instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/trl/trainer/ppo_trainer.py:273: UserWarning: No dataset is provided. Make sure to set config.batch_size to the correct value before training.
  warnings.warn(
[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Agent executed: . hello_world.sh


echo "Hello, World!"
Reward: 0.0 | Exit Code: -1


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


...

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Agent executed: bash hello_world.sh
Environment variables: Cannot find VI editor in this system.
Debug command: xargs ls # list all contents accessible to process
Environment variables
Reward: 0.0 | Exit Code: -1


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Agent executed: ./hello_world.sh
What is the output of this command when executed on a Linux system?
To get the user a Yes/No prompt without ending the shell
Reward: 0.0 | Exit Code: -1
Agent executed: touch hello_world.sh
nano hello_world.sh
echo "echo 'Hello World'" > hello_world.sh
source hello_world.sh

Design Pattern:Singleton
Reward: 0.0 | Exit Code: -1


ValueError: Batch size (1) does not match number of examples - but got 5 for: queries

In [2]:
!ls

k8s-agent-sandbox-gym.ipynb  tensorflow-tutorials
